In [7]:
import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

def validate_chunk_actions(chunk_actions):
    """
    Validates if there is significant average movement in the action chunk.
    
    Args:
        chunk_actions: List of lists containing action vectors
                      [forward/backward, left/right, look_horizontal, look_vertical]
    
    Returns:
        1 if significant average movement detected, 0 otherwise
    """
    # Convert chunk_actions to numpy array for easier calculations
    actions_array = np.array(chunk_actions)
    
    # Calculate average absolute movement for each action type
    avg_forward_back = np.mean(np.abs(actions_array[:, 0]))  # Forward/backward
    avg_left_right = np.mean(np.abs(actions_array[:, 1]))    # Left/right
    avg_look_horiz = np.mean(np.abs(actions_array[:, 2]))    # Horizontal rotation
    avg_look_vert = np.mean(np.abs(actions_array[:, 3]))     # Vertical rotation
    
    # Thresholds for average movement
    MOVE_THRESHOLD = 0.15    # Lower threshold for sustained movement
    ROTATION_THRESHOLD = 0.2  # Lower threshold for sustained rotation
    
    # Check if any average movement exceeds thresholds
    if (avg_forward_back > MOVE_THRESHOLD or 
        avg_left_right > MOVE_THRESHOLD or
        avg_look_horiz > ROTATION_THRESHOLD or 
        avg_look_vert > ROTATION_THRESHOLD):
        return 1
    
    return 0
# Define the folder path
folder_path = "/Users/vaibhavmishra/Desktop/Desktop/btx-game-aicode/clash_squad_agent_partitioned_features"

# Get all files in the folder
files = []
for file in os.listdir(folder_path):
    if file.endswith('.json'):
        files.append(file)

print(f"Found {len(files)} JSON files in the folder")

output_path = "/Users/vaibhavmishra/Desktop/Desktop/btx-game-aicode/clash_squad_partitioned_features_chunked/"
# Create output directories if they don't exist
os.makedirs(output_path, exist_ok=True)
os.makedirs(os.path.join(output_path, "features"), exist_ok=True)
os.makedirs(os.path.join(output_path, "actions"), exist_ok=True)

from tqdm import tqdm


for file in tqdm(files, desc="Processing files"):
    file_path = os.path.join(folder_path, file)
    file_name = file.split('.')[0]
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
            # print(f"Successfully read: {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")


    features = data['features']
    chunk_size = 20

    
    for i in range(0, len(features) - chunk_size, chunk_size):
        # Extract features from index i to i+chunk_size
        chunk_features = []
        for j in range(i, i + chunk_size):
            chunk_features.append(features[j][0])  # features[j][0] contains the feature vector
        
        # # Extract action from index i+chunk_size
        # if i + chunk_size < len(features):
        #     chunk_action = features[i + chunk_size][1]  # features[i+chunk_size][1] contains the action vector
        
        chunk_actions = []
        for j in range(i, i + chunk_size):
            action_feature = features[j][1]
            if action_feature[2] >= 10:
                action_feature[2] = 10
            if action_feature[2] <= -10:
                action_feature[2] = -10
            if action_feature[3] >= 10:
                action_feature[3] = 10
            if action_feature[3] <= -10:
                action_feature[3] = -10
            chunk_actions.append(action_feature)  # features[j][1] contains the action vector
        # print(chunk_features)
        # print(chunk_actions)
        validated = validate_chunk_actions(chunk_actions)
        
        # Convert to numpy arrays for efficient storage
        chunk_features_array = np.array(chunk_features, dtype=np.float32)
        chunk_action_array = np.array(chunk_actions, dtype=np.float32)
        

        
        # Save as .npy files
        features_filename = f"features/{file_name}_features_chunk_{i//chunk_size}.npy"
        actions_filename = f"actions/{file_name}_actions_chunk_{i//chunk_size}.npy"
        
        np.save(os.path.join(output_path, features_filename), chunk_features_array)
        np.save(os.path.join(output_path, actions_filename), chunk_action_array)
    

   



Found 3927 JSON files in the folder


Processing files:   0%|          | 0/3927 [00:00<?, ?it/s]

Processing files: 100%|██████████| 3927/3927 [07:17<00:00,  8.98it/s]
